# 03 — Functional Connectivity: Poster Figures

Generates publication-quality figures for the DBS Summit poster.

**Input:** `roi_ts_resampled`, `lfp_ts`, `target_sfreq` from Section 7 of `02_alignment_source_connectivity.ipynb`

**Figures produced:**
- `fig1_connectivity_matrices.png` — cGC + PSI matrices, high-beta vs low-beta
- `fig2_directionality_index.png` — Cortex → STN directed drive bar chart
- `fig3_pdc_spectrum.png`         — PDC frequency spectrum, cortex → STN
- `fig4_method_overview.png`      — All 4 methods × 2 bands summary heatmap

## 0. Imports & Setup

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sys.path.append('/workspace/src')

# Add upload directory so fc_pipeline can be imported
# Adjust this path if fc_pipeline.py is already installed in src/
FC_PIPELINE_DIR = '/workspace/notebooks/old_fc'   # change if needed
if FC_PIPELINE_DIR not in sys.path:
    sys.path.insert(0, FC_PIPELINE_DIR)

from fc_pipeline import FCMethods

RESULTS_DIR = Path('/workspace/shared/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Poster aesthetics ─────────────────────────────────────────
# Nature/Science journal style: clean, minimal, high contrast
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.8,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
})

# Color palette (colorblind-safe)
COL_HB   = '#D62728'   # high-beta: red
COL_LB   = '#1F77B4'   # low-beta:  blue
COL_SMA  = '#2CA02C'   # SMA:       green
COL_M1   = '#FF7F0E'   # M1:        orange
COL_STN  = '#9467BD'   # STN:       purple

print('✓ Setup complete')

ModuleNotFoundError: No module named 'frites'

## 1. Assemble Data for FCMethods

In [ ]:
# ── Assumes these variables exist from the alignment notebook ──
# roi_ts_resampled : dict  {roi_name: (n_epochs, n_times)}
# lfp_ts           : np.ndarray  (n_epochs, n_times)
# target_sfreq     : float  (250 Hz)

FS = target_sfreq   # sampling rate in Hz

# Use left-hemisphere sources (contralateral to STN)
n_epochs = min(
    roi_ts_resampled['SMA_L'].shape[0],
    roi_ts_resampled['M1_L'].shape[0],
    lfp_ts.shape[0]
)

# Stack into (n_epochs, n_channels, n_times)
# Channel order: 0=SMA, 1=M1, 2=STN
data_fc = np.stack([
    roi_ts_resampled['SMA_L'][:n_epochs],
    roi_ts_resampled['M1_L'][:n_epochs],
    lfp_ts[:n_epochs]
], axis=1)   # shape: (n_epochs, 3, n_times)

CH_NAMES = ['SMA', 'M1', 'STN']
N_CH     = len(CH_NAMES)

print(f'FC data shape: {data_fc.shape}  (epochs × channels × times)')
print(f'Channels: {CH_NAMES}')
print(f'Sampling rate: {FS} Hz')
print(f'Epoch duration: {data_fc.shape[2]/FS:.2f}s')

## 2. Run Connectivity Methods (High-beta & Low-beta)

In [ ]:
# ── Frequency bands ───────────────────────────────────────────
HB_MIN, HB_MAX = 21.0, 30.0   # high-beta Hz
LB_MIN, LB_MAX = 13.0, 21.0   # low-beta  Hz

# Normalized frequencies for ADTF/PDCoh (0–0.5 = 0–Nyquist)
hb_norm = np.linspace(HB_MIN / FS, HB_MAX / FS, 500)
lb_norm = np.linspace(LB_MIN / FS, LB_MAX / FS, 500)

fc = FCMethods(data_fc, fs=FS)

# ── Methods config ────────────────────────────────────────────
def build_methods(fmin, fmax, freqs_norm):
    return {
        'cGC':   {'maxlag': 10},
        'PSI':   {'fmin': fmin, 'fmax': fmax,
                  'band_width': 2.0, 'integrate': True},
        'PDCoh': {'model_order': 5, 'n_fft': 256,
                  'ica_method': 'infomax_extended', 'integrate': False},
        'DTF':   {'model_order': 5, 'n_fft': 256,
                  'ica_method': 'infomax_extended', 'integrate': False},
        'PLI':   {'fmin': fmin, 'fmax': fmax, 'integrate': True},
    }

print('Running high-beta (21–30 Hz) connectivity...')
results_hb = fc.compute_all(build_methods(HB_MIN, HB_MAX, hb_norm))
print('Running low-beta (13–21 Hz) connectivity...')
results_lb = fc.compute_all(build_methods(LB_MIN, LB_MAX, lb_norm))

print('✓ All connectivity methods complete')

In [ ]:
# ── Helper: extract beta-band slice from PDCoh/DTF (n_ch, n_ch, n_freqs) ──
def extract_band_mean(matrix_3d, fmin, fmax, n_fft=256):
    """Average PDCoh/DTF matrix over a specific frequency band."""
    nyquist  = FS / 2
    freqs_hz = np.linspace(0, nyquist, n_fft // 2)
    band_idx = (freqs_hz >= fmin) & (freqs_hz <= fmax)
    return matrix_3d[:, :, band_idx].mean(axis=2)   # (n_ch, n_ch)

def get_matrix(results, method, fmin=None, fmax=None):
    """Extract connectivity matrix; handle both integrated and non-integrated cases."""
    res = results.get(method, {})
    if 'error' in res:
        print(f'  Warning: {method} — {res["error"]}')
        return np.zeros((N_CH, N_CH))
    mat = res.get('matrix', None)
    if mat is None:
        return np.zeros((N_CH, N_CH))
    if mat.ndim == 3 and fmin is not None:   # non-integrated — average over band
        return extract_band_mean(mat, fmin, fmax)
    return mat

# Extract all matrices
cgc_hb  = get_matrix(results_hb, 'cGC')
cgc_lb  = get_matrix(results_lb, 'cGC')

psi_hb  = get_matrix(results_hb, 'PSI')
psi_lb  = get_matrix(results_lb, 'PSI')

pdc_hb  = get_matrix(results_hb, 'PDCoh', HB_MIN, HB_MAX)
pdc_lb  = get_matrix(results_lb, 'PDCoh', LB_MIN, LB_MAX)

dtf_hb  = get_matrix(results_hb, 'DTF',   HB_MIN, HB_MAX)
dtf_lb  = get_matrix(results_lb, 'DTF',   LB_MIN, LB_MAX)

pli_hb  = get_matrix(results_hb, 'PLI')
pli_lb  = get_matrix(results_lb, 'PLI')

print('Matrices extracted:')
for name, m in [('cGC_HB', cgc_hb), ('PSI_HB', psi_hb),
                ('PDC_HB', pdc_hb), ('DTF_HB', dtf_hb)]:
    print(f'  {name}: shape={m.shape}, max={m.max():.4f}')

## 3. Figure 1 — Connectivity Matrices (cGC & PSI, High-beta vs Low-beta)

In [ ]:
def plot_conn_matrix(ax, matrix, title, ch_names, cmap='RdBu_r',
                     highlight_row=None, highlight_col=None, vmax=None):
    """
    Plot a connectivity matrix as a heatmap.
    highlight_row/col: index of the row/column to outline (STN = 2).
    """
    n = matrix.shape[0]
    vmax = vmax or np.max(np.abs(matrix))

    im = ax.imshow(matrix, cmap=cmap, vmin=-vmax, vmax=vmax,
                   aspect='equal', interpolation='nearest')

    # Annotate values
    for i in range(n):
        for j in range(n):
            val = matrix[i, j]
            color = 'white' if abs(val) > vmax * 0.6 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')

    # Tick labels
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(ch_names, fontsize=9)
    ax.set_yticklabels(ch_names, fontsize=9)
    ax.set_xlabel('Target', fontsize=9)
    ax.set_ylabel('Source', fontsize=9)
    ax.set_title(title, fontsize=10, pad=6)

    # Highlight STN column (incoming connections to STN)
    if highlight_col is not None:
        rect = plt.Rectangle(
            (highlight_col - 0.5, -0.5), 1, n,
            linewidth=2.5, edgecolor='gold', facecolor='none'
        )
        ax.add_patch(rect)

    return im


# ── Build Figure 1 ────────────────────────────────────────────
fig1, axes = plt.subplots(2, 2, figsize=(9, 8))
fig1.suptitle(
    'Cortico-Subthalamic Connectivity: High-beta vs Low-beta',
    fontsize=13, fontweight='bold', y=1.01
)

pairs = [
    (axes[0, 0], cgc_hb,  f'cGC  |  High-beta ({HB_MIN:.0f}–{HB_MAX:.0f} Hz)'),
    (axes[0, 1], cgc_lb,  f'cGC  |  Low-beta  ({LB_MIN:.0f}–{LB_MAX:.0f} Hz)'),
    (axes[1, 0], psi_hb,  f'PSI  |  High-beta ({HB_MIN:.0f}–{HB_MAX:.0f} Hz)'),
    (axes[1, 1], psi_lb,  f'PSI  |  Low-beta  ({LB_MIN:.0f}–{LB_MAX:.0f} Hz)'),
]

for ax, mat, title in pairs:
    im = plot_conn_matrix(
        ax, mat, title, CH_NAMES,
        highlight_col=2,   # STN is channel 2
        vmax=None
    )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Band labels on left margin
fig1.text(0.01, 0.74, 'Conditional Granger Causality',
          va='center', rotation=90, fontsize=10, color='gray')
fig1.text(0.01, 0.27, 'Phase Slope Index',
          va='center', rotation=90, fontsize=10, color='gray')

# Gold box legend
gold_patch = mpatches.Patch(
    edgecolor='gold', facecolor='none', linewidth=2.5,
    label='STN input (cortex → STN)'
)
fig1.legend(handles=[gold_patch], loc='lower center',
            bbox_to_anchor=(0.5, -0.04), ncol=1, fontsize=9)

plt.tight_layout()
fig1.savefig(RESULTS_DIR / 'fig1_connectivity_matrices.png')
plt.show()
print('✓ Figure 1 saved')

## 4. Figure 2 — Directionality Index (Cortex → STN vs STN → Cortex)

In [ ]:
# ── Directionality index: cortex→STN minus STN→cortex ─────────
# STN index = 2, SMA = 0, M1 = 1
# matrix[i, j] = j → i  (row = target, col = source)

def directionality(mat, cortex_idx, stn_idx=2):
    """Compute DI = cortex→STN minus STN→cortex from connectivity matrix."""
    to_stn   = mat[stn_idx, cortex_idx]   # cortex → STN
    from_stn = mat[cortex_idx, stn_idx]   # STN → cortex
    return to_stn, from_stn, to_stn - from_stn

# Collect for all methods × bands × ROIs
methods_mat = {
    'cGC':  (cgc_hb,  cgc_lb),
    'PSI':  (psi_hb,  psi_lb),
    'PDCoh': (pdc_hb, pdc_lb),
    'DTF':  (dtf_hb,  dtf_lb),
}
rois       = [('SMA', 0), ('M1', 1)]
band_names = ['High-beta\n(21–30 Hz)', 'Low-beta\n(13–21 Hz)']


# ── Build Figure 2 ────────────────────────────────────────────
n_methods = len(methods_mat)
n_rois    = len(rois)

fig2, axes = plt.subplots(
    n_rois, n_methods, figsize=(12, 6),
    sharey='row', sharex=False
)
fig2.suptitle(
    'Directionality: Cortex → STN  vs  STN → Cortex',
    fontsize=13, fontweight='bold'
)

bar_w = 0.3
x     = np.array([0, 1])

for row_i, (roi_name, roi_idx) in enumerate(rois):
    for col_i, (method, (mat_hb, mat_lb)) in enumerate(methods_mat.items()):
        ax = axes[row_i, col_i]

        to_stn_hb, from_stn_hb, _ = directionality(mat_hb, roi_idx)
        to_stn_lb, from_stn_lb, _ = directionality(mat_lb, roi_idx)

        # Cortex→STN bars
        ax.bar(x - bar_w/2,
               [to_stn_hb, to_stn_lb],
               width=bar_w,
               color=[COL_HB, COL_LB], alpha=0.85,
               label='Cortex → STN')

        # STN→Cortex bars
        ax.bar(x + bar_w/2,
               [from_stn_hb, from_stn_lb],
               width=bar_w,
               color=[COL_HB, COL_LB], alpha=0.35,
               label='STN → Cortex', hatch='//')

        ax.set_xticks(x)
        ax.set_xticklabels(band_names, fontsize=8)
        ax.axhline(0, color='black', linewidth=0.6, linestyle='--')
        ax.grid(axis='y', alpha=0.3)

        if col_i == 0:
            ax.set_ylabel(f'{roi_name}\nConnectivity strength', fontsize=9)
        if row_i == 0:
            ax.set_title(method, fontsize=10, fontweight='bold')

# Shared legend
solid  = mpatches.Patch(color='gray', alpha=0.85, label='Cortex → STN')
hatch  = mpatches.Patch(color='gray', alpha=0.35,
                         hatch='//', label='STN → Cortex')
fig2.legend(
    handles=[solid, hatch], loc='lower center',
    bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=9
)

plt.tight_layout()
fig2.savefig(RESULTS_DIR / 'fig2_directionality_index.png')
plt.show()
print('✓ Figure 2 saved')

## 5. Figure 3 — PDC Frequency Spectrum (Cortex → STN)

In [ ]:
# Re-run PDCoh with integrate=False to get full spectrum
print('Computing full-spectrum PDC...')
results_spectrum = fc.compute_all({
    'PDCoh': {
        'model_order': 5, 'n_fft': 256,
        'ica_method': 'infomax_extended', 'integrate': False
    },
    'DTF': {
        'model_order': 5, 'n_fft': 256,
        'ica_method': 'infomax_extended', 'integrate': False
    },
})

# Frequency axis
N_FFT  = 256
freqs_hz = np.linspace(0, FS / 2, N_FFT // 2)

pdc_full = results_spectrum['PDCoh']['matrix']   # (n_ch, n_ch, n_freqs)
dtf_full = results_spectrum['DTF']['matrix']     # (n_ch, n_ch, n_freqs)

# Focus on 0–50 Hz
freq_mask = freqs_hz <= 50
f_plot    = freqs_hz[freq_mask]

# ── Build Figure 3 ────────────────────────────────────────────
fig3, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
fig3.suptitle(
    'Frequency-Resolved Directed Connectivity  |  Cortex → STN  &  STN → Cortex',
    fontsize=12, fontweight='bold'
)

# matrix[target, source, freq] → source→target
plot_pairs = [
    # (ax,          matrix,   source_idx, target_idx, label,        color)
    (axes[0, 0], pdc_full, 0, 2, 'SMA → STN  (PDC)', COL_SMA),
    (axes[0, 0], pdc_full, 2, 0, 'STN → SMA  (PDC)', COL_STN),
    (axes[0, 1], pdc_full, 1, 2, 'M1  → STN  (PDC)', COL_M1),
    (axes[0, 1], pdc_full, 2, 1, 'STN → M1   (PDC)', COL_STN),
    (axes[1, 0], dtf_full, 0, 2, 'SMA → STN  (DTF)', COL_SMA),
    (axes[1, 0], dtf_full, 2, 0, 'STN → SMA  (DTF)', COL_STN),
    (axes[1, 1], dtf_full, 1, 2, 'M1  → STN  (DTF)', COL_M1),
    (axes[1, 1], dtf_full, 2, 1, 'STN → M1   (DTF)', COL_STN),
]

added_axes = set()
for ax, mat, src, tgt, label, col in plot_pairs:
    ls = '-' if 'SMA' in label or 'M1' in label else '--'
    ax.plot(f_plot, mat[tgt, src, freq_mask],
            color=col, lw=1.8, ls=ls, label=label)

for ax in axes.flat:
    # Shade beta bands
    ax.axvspan(LB_MIN, LB_MAX, color=COL_LB, alpha=0.10, label='Low-beta')
    ax.axvspan(HB_MIN, HB_MAX, color=COL_HB, alpha=0.10, label='High-beta')
    ax.set_xlim(1, 50)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Connectivity strength', fontsize=9)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc='upper right')

axes[1, 0].set_xlabel('Frequency (Hz)', fontsize=10)
axes[1, 1].set_xlabel('Frequency (Hz)', fontsize=10)

# Beta band annotations
for ax in axes.flat:
    ax.annotate('low-β', xy=((LB_MIN+LB_MAX)/2, 0.97),
                ha='center', fontsize=7, color=COL_LB)
    ax.annotate('high-β', xy=((HB_MIN+HB_MAX)/2, 0.97),
                ha='center', fontsize=7, color=COL_HB)

plt.tight_layout()
fig3.savefig(RESULTS_DIR / 'fig3_pdc_spectrum.png')
plt.show()
print('✓ Figure 3 saved')

## 6. Figure 4 — Summary Heatmap (All Methods × 2 Bands)

In [ ]:
# ── Summarize cortex→STN directionality across all methods ────
# DI = (cortex→STN) − (STN→cortex)  [signed, positive = feedforward]

method_labels = ['cGC', 'PSI', 'PDCoh', 'DTF', 'PLI']
roi_labels    = ['SMA', 'M1']
band_labels   = ['High-beta\n21–30 Hz', 'Low-beta\n13–21 Hz']

mats_hb = {
    'cGC': cgc_hb, 'PSI': psi_hb, 'PDCoh': pdc_hb,
    'DTF': dtf_hb, 'PLI': pli_hb
}
mats_lb = {
    'cGC': cgc_lb, 'PSI': psi_lb, 'PDCoh': pdc_lb,
    'DTF': dtf_lb, 'PLI': pli_lb
}

# Build DI table: rows = method, cols = [SMA_HB, SMA_LB, M1_HB, M1_LB]
n_m = len(method_labels)
n_r = len(roi_labels)
n_b = len(band_labels)

di_table = np.zeros((n_m, n_r, n_b))

for mi, mname in enumerate(method_labels):
    for ri, (rname, ridx) in enumerate(zip(roi_labels, [0, 1])):
        _, _, di_hb = directionality(mats_hb[mname], ridx)
        _, _, di_lb = directionality(mats_lb[mname], ridx)
        di_table[mi, ri, 0] = di_hb
        di_table[mi, ri, 1] = di_lb

# Reshape to (n_methods, n_rois * n_bands)
# Columns: SMA-HB, SMA-LB, M1-HB, M1-LB
di_flat = di_table.reshape(n_m, n_r * n_b)
col_labels = ['SMA\nHigh-β', 'SMA\nLow-β', 'M1\nHigh-β', 'M1\nLow-β']

# ── Build Figure 4 ────────────────────────────────────────────
fig4, ax = plt.subplots(figsize=(8, 5))

vmax = np.max(np.abs(di_flat)) if np.max(np.abs(di_flat)) > 0 else 1
im   = ax.imshow(di_flat, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                 aspect='auto')

# Annotate
for i in range(n_m):
    for j in range(n_r * n_b):
        val   = di_flat[i, j]
        color = 'white' if abs(val) > vmax * 0.6 else 'black'
        symbol = '▲' if val > 0 else '▼'
        ax.text(j, i, f'{symbol}\n{val:.3f}',
                ha='center', va='center',
                fontsize=8.5, color=color, fontweight='bold')

ax.set_xticks(range(n_r * n_b))
ax.set_xticklabels(col_labels, fontsize=9)
ax.set_yticks(range(n_m))
ax.set_yticklabels(method_labels, fontsize=10, fontweight='bold')
ax.set_xlabel('ROI × Frequency Band', fontsize=10)
ax.set_title(
    'Directionality Index  (▲ Cortex → STN  |  ▼ STN → Cortex)\n'
    'Positive = feedforward drive via hyperdirect pathway',
    fontsize=11, fontweight='bold'
)

# Vertical separator between SMA and M1 groups
ax.axvline(1.5, color='white', linewidth=2.5)

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02,
             label='Directionality Index')

plt.tight_layout()
fig4.savefig(RESULTS_DIR / 'fig4_method_overview.png')
plt.show()
print('✓ Figure 4 saved')

In [ ]:
print('All figures saved to:', RESULTS_DIR)
for f in sorted(RESULTS_DIR.glob('fig*.png')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s}  {size_kb:.0f} KB')